# Soil Analytics – Model Training

In [1]:
import pandas as pd, joblib, os
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report

PROC = "processed"
MODELS = "../models"
os.makedirs(MODELS, exist_ok=True)

## Crop model

In [2]:
crop = pd.read_csv(f"{PROC}/crop_clean.csv")
le_c = joblib.load(f"{MODELS}/crop_label_encoder.pkl")
pre_c = joblib.load(f"{MODELS}/crop_preprocessor.pkl")

X, y = crop.drop(columns="label"), le_c.transform(crop["label"])
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=.2, stratify=y, random_state=42)
X_tr, X_te = pre_c.transform(X_tr), pre_c.transform(X_te)

gs = GridSearchCV(RandomForestClassifier(random_state=42),
                  {"n_estimators": [100, 200, 300], "max_depth": [None, 20], "min_samples_leaf": [1, 2]},
                  cv=5, scoring="f1_macro", n_jobs=-1).fit(X_tr, y_tr)
crop_model = gs.best_estimator_
pred = crop_model.predict(X_te)
print(gs.best_params_)
print("accuracy:", round(accuracy_score(y_te, pred), 4), "| macro-F1:", round(f1_score(y_te, pred, average="macro"), 4))

{'max_depth': None, 'min_samples_leaf': 2, 'n_estimators': 300}
accuracy: 0.9932 | macro-F1: 0.9932


## Fertilizer model

In [3]:
fert = pd.read_csv(f"{PROC}/fert_clean.csv")
le_f = joblib.load(f"{MODELS}/fert_label_encoder.pkl")
pre_f = joblib.load(f"{MODELS}/fert_preprocessor.pkl")

X, y = fert.drop(columns="Fertilizer Name"), le_f.transform(fert["Fertilizer Name"])
strat = y if pd.Series(y).value_counts().min() >= 2 else None
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=.2, stratify=strat, random_state=42)
X_tr, X_te = pre_f.transform(X_tr), pre_f.transform(X_te)

gs = GridSearchCV(RandomForestClassifier(random_state=42),
                  {"n_estimators": [100, 200], "max_depth": [None, 5, 10], "min_samples_leaf": [1, 2]},
                  cv=3, scoring="accuracy", n_jobs=-1).fit(X_tr, y_tr)
fert_model = gs.best_estimator_
pred = fert_model.predict(X_te)
print(gs.best_params_)
print("accuracy:", round(accuracy_score(y_te, pred), 4))
print(classification_report(y_te, pred, labels=range(len(le_f.classes_)), target_names=le_f.classes_, zero_division=0))

{'max_depth': 10, 'min_samples_leaf': 1, 'n_estimators': 200}
accuracy: 1.0
              precision    recall  f1-score   support

    10-26-26       1.00      1.00      1.00         1
    14-35-14       1.00      1.00      1.00         3
    17-17-17       1.00      1.00      1.00         1
       20-20       1.00      1.00      1.00         3
       28-28       1.00      1.00      1.00         3
         DAP       1.00      1.00      1.00         4
        Urea       1.00      1.00      1.00         5

    accuracy                           1.00        20
   macro avg       1.00      1.00      1.00        20
weighted avg       1.00      1.00      1.00        20



## Export

In [4]:
joblib.dump(crop_model, f"{MODELS}/crop_model.pkl")
joblib.dump(fert_model, f"{MODELS}/fert_model.pkl")
print(sorted(os.listdir(MODELS)))

['crop_label_encoder.pkl', 'crop_model.pkl', 'crop_preprocessor.pkl', 'fert_label_encoder.pkl', 'fert_model.pkl', 'fert_preprocessor.pkl']
